In [1]:
# Import the basic spark library
from pyspark.sql import SparkSession

# Create an entry point to the PySpark Application
spark = SparkSession.builder \
      .master("local") \
      .appName("ProjSMBUD") \
      .getOrCreate()
# master contains the URL of your remote spark instance or 'local'

In [2]:
  # Load DataFrames (we prefer directly them to the rdd, because we already had csv with headers)
#Configuring them with some examples to check the correctness (now commented before and after the casting)

df_incollection = spark.read.option("header", True).option("delimiter", ";")\
                .csv("incollectionFormatted.csv.txt")
df_book = spark.read.option("header", True).option("delimiter", ";")\
                .csv("bookFormatted.csv.txt")
df_author = spark.read.option("header", True).option("delimiter", ",")\
                .csv("authorFormatted.csv.txt")

#Load bridge tables? (No, now they are in the entities), #they had already id string types, but now we don't have them





# We use of datatypes from pyspark.sql.types
from pyspark.sql.types import StringType, DateType, FloatType, StructType, StructField, ArrayType
from pyspark.sql.functions import * 


##We use the ids in string format!
##TO HAVE THE ARRAY OF STRINGS
df_book = df_book\
    .select("id","date", "url", "isbn", "citationscore", "publisher" , "title", split(col("editor_array"),",").alias("editor_array"))
df_incollection = df_incollection\
    .select("id","title","date", "url", "pages", "citationscore", \
    "book_id", split(col("author_array"),",").alias("author_array"), \
    split(col("cited_inc_array"),",").alias("cited_inc_array"))


#df_incollection.limit(5).select("id", 'author_array').show()
df_incollection = df_incollection \
  .withColumn("id" ,
              df_incollection["id"]
              .cast(StringType()))   \
  .withColumn("title"  ,
              df_incollection["title"]
              .cast(StringType())) \
  .withColumn("date",
              df_incollection["date"]
              .cast(DateType()))    \
  .withColumn("url"  ,
              df_incollection["url"]
              .cast(StringType())) \
  .withColumn("pages"  ,
              df_incollection["pages"]
              .cast(StringType())) \
  .withColumn("citationscore"  ,
              df_incollection["citationscore"]
              .cast(FloatType())) \
#  .withColumn("author_array"  ,
#              df_incollection["author_array"]
#              .cast(ArrayType(StringType()))) \
#  .withColumn("cited_inc_array"  ,
#              df_incollection["cited_inc_array"]
#              .cast(ArrayType(StringType()))) \
#df_incollection.printSchema()
#df_incollection.filter(array_contains(df_incollection.author_array,"7370")).select("id","author_array").show(truncate = False)
#5371|[7371,  7370,  7372]
#df_incollection.select(explode(df_incollection.cited_inc_array)).show()
df_incollection.printSchema()
df_incollection.limit(5).show()


#df_book.printSchema()
#df_book.show()
df_book = df_book \
  .withColumn("id" ,
              df_book["id"]
              .cast(StringType()))   \
  .withColumn("isbn"  ,
              df_book["isbn"]
              .cast(StringType())) \
  .withColumn("date",
              df_book["date"]
              .cast(DateType()))    \
  .withColumn("url"  ,
              df_book["url"]
              .cast(StringType())) \
  .withColumn("publisher"  ,
              df_book["publisher"]
              .cast(StringType())) \
  .withColumn("citationscore"  ,
              df_book["citationscore"]
              .cast(FloatType())) \
  .withColumn("title"  ,
              df_book["title"]
              .cast(StringType())) \
#  .withColumn("editors_array"  ,
#              df_book_editors["editors_array"]) \
df_book.printSchema()
#df_book.select(explode(df_book.editor_array)).show()
df_book.limit(5).show()

#df_author.printSchema()
#df_author.limit(5).show()
df_author = df_author \
  .withColumn("id" ,
              df_author["id"]
              .cast(StringType()))   \
  .withColumn("name"  ,
              df_author["name"]
              .cast(StringType())) \
  .withColumn("citationscore",
              df_author["citationscore"]
              .cast(FloatType()))    \
#email is already a string
df_author.printSchema() 
df_author.limit(5).show()



root
 |-- id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- date: date (nullable = true)
 |-- url: string (nullable = true)
 |-- pages: string (nullable = true)
 |-- citationscore: float (nullable = true)
 |-- book_id: string (nullable = true)
 |-- author_array: array (nullable = true)
 |    |-- element: string (containsNull = false)
 |-- cited_inc_array: array (nullable = true)
 |    |-- element: string (containsNull = false)

+----+--------------------+----------+--------------------+---------+-------------+-------+------------------+--------------------+
|  id|               title|      date|                 url|    pages|citationscore|book_id|      author_array|     cited_inc_array|
+----+--------------------+----------+--------------------+---------+-------------+-------+------------------+--------------------+
|5367|Ear Shape for Bio...|2017-07-12|https://doi.org/1...|  372-378|    0.8379688|   5169|            [7367]|[6264, 7146, 7146...|
|5368|      ARP Poi

In [3]:
#- WHERE, GROUP BY, HAVING, 1 JOIN
#This query searches the authors that starts with "Paul", then, finds the incollections which has these authors 
#and returns the book_id that have a total number of these(filtered authors) less than 3.
#The incollections with null author_array are filtered.
df_auth_filt =df_author.where(col("name").startswith("Paul")).select(col("id"),col("name"))
df_auth_filt.limit(10).show()

exploded_df = df_incollection.filter(col("author_array").isNotNull())\
                     .select(col("id"),col("book_id"),explode(col("author_array")))\
                     .withColumnRenamed("col", "author")\
                     .sort(col("author").asc())
exploded_df.limit(10).show()

exploded_df.join(df_auth_filt, exploded_df.author == df_auth_filt.id,"inner")\
           .groupBy("book_id")\
           .count()\
           .withColumnRenamed("count","Number of authors")\
           .where(col("count")<3)\
           .sort(col("Number of authors").desc())\
           .show()


+----+-----------------+
|  id|             name|
+----+-----------------+
|2000|      Paul Kocher|
|2337|     Paul Ferring|
|2359| Paul B. Hermanns|
|2853| Paul B. Thompson|
|3123|       Paul Tubig|
|3168|     Paul Firenze|
|3211|  Paul Jones 0001|
|3244|     Paul Partlow|
|3356|    Paul Harrison|
|3698|Paul J. Laurienti|
+----+-----------------+

+----+-------+------+
|  id|book_id|author|
+----+-------+------+
|7065|   5134|  2108|
|7307|   5067|  2128|
|5689|   5097|  2252|
|6028|   5124|  2252|
|7058|   5121|  3012|
|7057|   5131|  3319|
|7073|   5148|  3809|
|7042|   5067|  3842|
|5791|   5084|  4446|
|7017|   5130|  4880|
+----+-------+------+

+-------+-----------------+
|book_id|Number of authors|
+-------+-----------------+
|   5123|                2|
|   5111|                2|
|   5170|                1|
|   5069|                1|
|   5103|                1|
|   5119|                1|
+-------+-----------------+



In [4]:
#GROUP BY, 1 JOIN, AS 
#This query returns the number of authors of a book which are also editor of an incollection of that book.
#The books and incollections with null editor_array or author_array are filtered.
#books
exploded_df = df_book.filter(col("editor_array").isNotNull())\
                     .select(col("id"),col("title"),explode(col("editor_array")))\
                     .withColumnRenamed("col", "editor")
exploded_df.limit(10).show()
#incollections
df_incol_filt=df_incollection.filter(col("author_array").isNotNull())\
                             .select(col("book_id"),col("id"),col("author_array"))
df_incol_filt.limit(10).show()
#join
new_df=exploded_df.join(df_incol_filt,exploded_df.id==df_incol_filt.book_id,"inner")\
           .filter(array_contains(df_incol_filt.author_array, exploded_df.editor))
new_df.show()
new_df.groupBy("title")\
      .agg(count("title").alias("Number of Author\Editor")).show()
      #Alternative
      #.count().withColumnRenamed("count", "Number of Author\Editor").show()
      



+----+--------------------+------+
|  id|               title|editor|
+----+--------------------+------+
|5069|Encyclopedia of C...|  5180|
|5070|Encyclopedia of C...|  5052|
|5070|Encyclopedia of C...|  5180|
|5071|Handbook of Ambie...|  5182|
|5071|Handbook of Ambie...|  5187|
|5071|Handbook of Ambie...|  5186|
|5071|Handbook of Ambie...|  5185|
|5071|Handbook of Ambie...|  5181|
|5071|Handbook of Ambie...|  5184|
|5071|Handbook of Ambie...|  5183|
+----+--------------------+------+

+-------+----+------------------+
|book_id|  id|      author_array|
+-------+----+------------------+
|   5169|5367|            [7367]|
|   5109|5369|            [7368]|
|   5114|5370|            [7369]|
|   5078|5371|[7371, 7370, 7372]|
|   5119|5372|            [7368]|
|   5177|5376|            [7373]|
|   5176|5377|            [7374]|
|   5131|5378|            [7375]|
|   5103|5379|            [7376]|
|   5074|5381|            [7377]|
+-------+----+------------------+

+----+--------------------+-----

In [5]:
#UPDATE
#This command update the author array by adding the author with id 2000 in the incollections that have:  
#citationscore greater than 1.7
#null author array 
#In this scenario, we are assuming that the author we are adding in the incollections is already contained in the db.
#author assumption
df_author.filter(col("id")=="2000").show()
#update
df_incollection.filter((col("citationscore")>1.7) & (col("author_array").isNull())).show()

df_incollection=df_incollection.withColumn("author_array",\
                when((col("citationscore")>1.7) & (col("author_array").isNull()), array(lit("2000"))).otherwise(col("author_array")))
#proof
df_incollection.filter((col("citationscore")>1.7) & (col("author_array").isNull())).show()

df_incollection.filter(col("citationscore")>1.7).sort(col("citationscore").desc()).show()


+----+-----------+-------------+--------------------+
|  id|       name|citationscore|               email|
+----+-----------+-------------+--------------------+
|2000|Paul Kocher|     0.977108|paul.kocher@gmail...|
+----+-----------+-------------+--------------------+

+----+--------------------+----------+--------------------+-----+-------------+-------+------------+--------------------+
|  id|               title|      date|                 url|pages|citationscore|book_id|author_array|     cited_inc_array|
+----+--------------------+----------+--------------------+-----+-------------+-------+------------+--------------------+
|6213|Time-Based Access...|2017-07-12|https://doi.org/1...| 1297|    1.8360946|   5130|        null|[7036, 7068, 7318...|
|6296|Public-Key Encryp...|2017-07-12|https://doi.org/1...| 1001|    2.1130114|   5128|        null|[5984, 6698, 6191...|
|6345|Fixed Window Expo...|2017-07-12|https://doi.org/1...|  482|    1.7093747|   5132|        null|[7061, 7072, 7337..